# 09 — The matching model (Phase 9)

## Concept: supervised classification
Each candidate pair is one **example**. Its **features** (Phase 8) are the inputs and its **label** is the answer:
`1` = same business, `0` = different business. A classifier learns from labelled examples a function
`features → probability of "same business"`.

## Where do the training examples come from?
From the candidates that blocking produced for the 20,000 **tuning** entities (train split):
- **positives** = candidates that are true matches;
- **negatives** = all other candidates. Because blocking already picked records that *look* similar,
  these are **hard negatives** (same name in another town, same street but another business…) — exactly the
  mistakes we must learn to avoid. Random negatives would be far too easy.

## How we evaluate without leaking
1. **Cross-validation on the tuning entities** (3 folds, split by *entity*, so all candidates of one entity are in the
   same fold) gives an honest probability for every tuning pair → we choose the decision **threshold** there.
2. Train the model on **all** tuning pairs, predict the **validation** entities' pairs, apply that threshold,
   and compute the official macro F0.5. Validation labels are used **only** for this final score.

## Models, from simple to strong
| model | idea |
|---|---|
| Logistic regression | a weighted sum of the features → probability. Simple, interpretable. |
| Random forest | many decision trees, each on a random sample; they vote. |
| Gradient boosting (`HistGradientBoostingClassifier`) | trees added one after another, each fixing the previous ones' mistakes. Usually the strongest on tabular data; handles missing values (NaN) itself. |

All three are in scikit-learn (BSD licence), tiny compared to the 8B-parameter limit.

In [1]:
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.evaluation import score_per_entity, score_report
from src.features import FEATURE_COLUMNS
from src.rule_experiment import load_query_sets

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 60)
start = time.time()

In [2]:
pairs = pd.read_parquet(PROJECT_ROOT / "cache" / "phase8_features.parquet")
_, truth = load_query_sets()

train = pairs[pairs["query_set"] == "tuning"].reset_index(drop=True)
valid = pairs[pairs["query_set"] == "validation"].reset_index(drop=True)
X_train, y_train = train[FEATURE_COLUMNS].astype("float32"), train["label"].to_numpy()
X_valid, y_valid = valid[FEATURE_COLUMNS].astype("float32"), valid["label"].to_numpy()

print(f"train pairs: {len(train):,}  positives {y_train.mean():.1%}  entities {train['source1_entity_id'].nunique():,}")
print(f"valid pairs: {len(valid):,}  positives {y_valid.mean():.1%}  entities {valid['source1_entity_id'].nunique():,}")
print(f"features: {len(FEATURE_COLUMNS)}")

train pairs: 998,797  positives 6.5%  entities 20,000
valid pairs: 999,057  positives 6.5%  entities 20,000
features: 34


## From probabilities to a submission, and its score

`to_predictions` keeps every pair whose probability is ≥ the threshold and builds the submission format.
Entities with no pair above the threshold get an empty list (predicted singleton).

In [3]:
def to_predictions(pair_table, probabilities, threshold):
    chosen = pair_table.loc[probabilities >= threshold, ["source1_entity_id", "candidate_id"]]
    return chosen.groupby("source1_entity_id")["candidate_id"].agg(",".join).rename("matched_entity_ids").reset_index()

THRESHOLDS = np.round(np.arange(0.05, 0.96, 0.05), 2)

def best_threshold(pair_table, probabilities, truth_table):
    scores = [score_report(to_predictions(pair_table, probabilities, t), truth_table)["macro_F0.5"] for t in THRESHOLDS]
    return float(THRESHOLDS[int(np.argmax(scores))]), max(scores), scores

## The evaluation routine (same for every model)

In [4]:
def evaluate(name, make_model):
    t0 = time.time()
    # 1) out-of-fold probabilities on the tuning entities (3 folds grouped by entity)
    oof = np.zeros(len(train), dtype="float64")
    for fit_idx, pred_idx in GroupKFold(n_splits=3).split(X_train, y_train, groups=train["source1_entity_id"]):
        model = make_model().fit(X_train.iloc[fit_idx], y_train[fit_idx])
        oof[pred_idx] = model.predict_proba(X_train.iloc[pred_idx])[:, 1]
    threshold, tuning_f05, curve = best_threshold(train, oof, truth["tuning"])

    # 2) final model on all tuning pairs -> validation
    model = make_model().fit(X_train, y_train)
    p_valid = model.predict_proba(X_valid)[:, 1]
    report = score_report(to_predictions(valid, p_valid, threshold), truth["validation"])
    row = {
        "model": name,
        "oof AUC": roc_auc_score(y_train, oof),
        "threshold (chosen on tuning)": threshold,
        "tuning F0.5 (out-of-fold)": tuning_f05,
        "validation AUC": roc_auc_score(y_valid, p_valid),
        "validation F0.5": report["macro_F0.5"],
        "validation F0.5 matched entities": report["F0.5 on entities with matches"],
        "validation F0.5 singletons": report["F0.5 on singletons"],
        "pair precision": report["pair precision"],
        "pair recall": report["pair recall"],
        "seconds": round(time.time() - t0),
    }
    print(f"{name}: validation F0.5 = {row['validation F0.5']:.4f}  ({row['seconds']}s)")
    return row, model, oof, p_valid, curve

---
## Model 1 — Logistic regression

Logistic regression cannot handle missing values (NaN: e.g. address features when the candidate has no address),
so a small pipeline first:
1. `SimpleImputer(add_indicator=True)` — fill NaN with the median **and** add a 0/1 column "this was missing";
2. `StandardScaler` — put every feature on the same scale (mean 0, std 1) so the weights are comparable;
3. `LogisticRegression`.

In [5]:
results, models, oof_probs, valid_probs, curves = [], {}, {}, {}, {}

def logistic():
    return make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                         LogisticRegression(max_iter=2000, C=1.0))

row, models["logistic"], oof_probs["logistic"], valid_probs["logistic"], curves["logistic"] = evaluate("logistic regression", logistic)
results.append(row)

logistic regression: validation F0.5 = 0.8886  (53s)


### What did logistic regression learn?

After scaling, each weight says how much one standard deviation of that feature pushes the prediction
towards "match" (positive) or "no match" (negative).

In [6]:
lr = models["logistic"]
names = FEATURE_COLUMNS + [f"{FEATURE_COLUMNS[i]}_missing" for i in lr[0].indicator_.features_]
weights = pd.Series(lr[-1].coef_[0], index=names).sort_values()
pd.concat([weights.head(8), weights.tail(10)]).round(2).to_frame("weight")

,weight
legal_conflict,-1.43
core_token_set,-1.39
name_ratio,-1.37
block_rank,-1.32
addr_token_sort,-0.87
name_len_diff,-0.50
name_jaccard,-0.41
num_conflict,-0.26
num_c_subset,0.49
num_jaccard_missing,0.75


**Careful when reading these weights.** The strongest positive weights are the ones we expected
(`addr_token_set`, `name_tfidf`, `core_joined_ratio`, `num_jaccard`), and `legal_conflict` and `block_rank` push strongly towards "no match".
But some name-similarity features get *negative* weights (`core_token_set`, `name_ratio`). That doesn't mean "similar names = different
business": these features are highly **correlated** with each other, so the model splits and offsets the credit between them.
With correlated features, read the **group**, not a single sign.

---
## Model 2 — Random forest

`max_samples=0.3` (each tree sees 30% of the rows) and `min_samples_leaf=20` keep memory and time reasonable on 1M rows.
Random forest in scikit-learn does not accept NaN in every version, so we fill NaN with −1 (a value no similarity takes).

In [7]:
def random_forest():
    return make_pipeline(SimpleImputer(strategy="constant", fill_value=-1),
                         RandomForestClassifier(n_estimators=100, max_depth=18, min_samples_leaf=20, max_samples=0.3,
                                                n_jobs=-1, random_state=0))

row, models["forest"], oof_probs["forest"], valid_probs["forest"], curves["forest"] = evaluate("random forest", random_forest)
results.append(row)

random forest: validation F0.5 = 0.9168  (116s)


---
## Model 3 — Gradient boosting

`HistGradientBoostingClassifier` bins every feature into ≤ 255 buckets, which makes it fast on millions of rows,
and it handles NaN natively (it learns which side of each split missing values should go).

In [8]:
def gradient_boosting():
    return HistGradientBoostingClassifier(max_iter=400, learning_rate=0.1, max_leaf_nodes=63, min_samples_leaf=50,
                                          l2_regularization=1.0, early_stopping=False, random_state=0)

row, models["boosting"], oof_probs["boosting"], valid_probs["boosting"], curves["boosting"] = evaluate("gradient boosting", gradient_boosting)
results.append(row)

gradient boosting: validation F0.5 = 0.9340  (204s)


---
## Comparison

In [9]:
comparison = pd.DataFrame(results).set_index("model")
comparison.loc["Jaccard rule (Phase 7)", ["validation F0.5", "threshold (chosen on tuning)"]] = [0.7058, 0.60]
comparison.round(4)

,oof AUC,threshold (chosen on tuning),tuning F0.5 (out-of-fold),validation AUC,validation F0.5,validation F0.5 matched entities,validation F0.5 singletons,pair precision,pair recall,seconds
model,,,,,,,,,,
logistic regression,0.9973,0.65,0.8874,0.9973,0.8886,0.8928,0.8200,0.9537,0.8110,53.0
random forest,0.9986,0.55,0.9135,0.9987,0.9168,0.9188,0.8843,0.9687,0.8476,116.0
gradient boosting,0.9993,0.70,0.9324,0.9994,0.9340,0.9353,0.9130,0.9785,0.8714,204.0
Jaccard rule (Phase 7),NaN,0.60,NaN,NaN,0.7058,NaN,NaN,NaN,NaN,NaN


### F0.5 vs threshold (out-of-fold, tuning entities)

How sensitive is each model to the threshold? A flat curve near the top is safer than a sharp peak.

In [10]:
pd.DataFrame(curves, index=THRESHOLDS).round(4)

,logistic,forest,boosting
0.05,0.7192,0.6738,0.8450
0.10,0.7629,0.7489,0.8759
0.15,0.7911,0.7992,0.8916
0.20,0.8139,0.8309,0.9025
0.25,0.8317,0.8583,0.9098
0.30,0.8448,0.8815,0.9153
0.35,0.8561,0.8959,0.9193
0.40,0.8651,0.9054,0.9229
0.45,0.8726,0.9104,0.9258
0.50,0.8784,0.9126,0.9281


Gradient boosting is the best at *every* threshold, and its curve is flat between 0.55 and 0.80 (all within 0.002 of the peak) —
the choice of threshold is not fragile. Logistic regression peaks lower, random forest drops fast above 0.60.

---
## Looking inside the best model

In [11]:
best = comparison.drop(index="Jaccard rule (Phase 7)")["validation F0.5"].idxmax()
key = {"logistic regression": "logistic", "random forest": "forest", "gradient boosting": "boosting"}[best]
threshold = comparison.loc[best, "threshold (chosen on tuning)"]
p = valid_probs[key]
print("best model:", best, "| threshold", threshold)

cm = confusion_matrix(y_valid, (p >= threshold).astype(int))
pd.DataFrame(cm, index=["true: different", "true: same"], columns=["predicted: different", "predicted: same"])

best model: gradient boosting | threshold 0.7


,predicted: different,predicted: same
true: different,932917,1322
true: same,4704,60114


**Confusion matrix (pair level, validation):** bottom-right = correct matches, top-right = **false merges**
(what F0.5 punishes most), bottom-left = missed matches among the candidates.

In [12]:
per_entity = score_per_entity(to_predictions(valid, p, threshold), truth["validation"])
per_entity["kind"] = np.select([per_entity["n_true"] == 0, per_entity["n_true"] <= 2, per_entity["n_true"] <= 5],
                               ["singleton", "1-2 matches", "3-5 matches"], "6+ matches")
per_entity.groupby("kind").agg(entities=("f05", "size"), mean_F05=("f05", "mean"),
                               avg_true=("n_true", "mean"), avg_predicted=("n_pred", "mean")).round(3)

,entities,mean_F05,avg_true,avg_predicted
kind,,,,
1-2 matches,4476,0.900,1.753,1.611
3-5 matches,12123,0.944,3.843,3.413
6+ matches,2251,0.955,6.463,5.655
singleton,1150,0.913,0.000,0.103


- **Singletons** (no true match) now score **0.913**: the model predicts an average of only 0.10 IDs for them.
  With the Jaccard rule this was ~0.5.
- Entities with 1–2 matches are the hardest (0.900): one mistake or one miss is a large share of their list.
- The model predicts slightly **fewer** IDs than exist (3.41 vs 3.84 for 3–5-match entities) — the precision-first behaviour F0.5 rewards.

### Which features does the model rely on? (permutation importance)

Shuffle one feature's values and see how much the model's AUC drops: a big drop = the model relies on it.
Computed on a 100k-pair sample of the validation pairs (analysis only — nothing is tuned on it).

In [13]:
sample = np.random.default_rng(0).choice(len(valid), 100_000, replace=False)
importance = permutation_importance(models[key], X_valid.iloc[sample], y_valid[sample], scoring="roc_auc",
                                    n_repeats=3, random_state=0, n_jobs=1)
pd.Series(importance.importances_mean, index=FEATURE_COLUMNS).sort_values(ascending=False).head(15).round(4).to_frame("AUC drop")

,AUC drop
num_jaccard,0.0018
name_tfidf,0.0012
name_jaccard,0.0012
addr_tfidf,0.0010
core_joined_ratio,0.0009
addr_token_set,0.0009
addr_jaccard,0.0008
core_partial,0.0006
block_rank,0.0006
legal_conflict,0.0005


The AUC drops are small because the model is already near-perfect at *ranking* (AUC 0.9994) and many features are redundant —
removing one, the others compensate. The model leans on **house-number agreement**, **name similarity (TF-IDF, Jaccard, joined names)**,
**address similarity**, the **blocking rank** and the **legal-form conflict** — all signals we discovered by hand in Phases 3–8.

In [14]:
out = valid[["source1_entity_id", "candidate_id", "label"]].copy()
out["probability"] = p
out["query_set"] = "validation"
oof = train[["source1_entity_id", "candidate_id", "label"]].copy()
oof["probability"] = oof_probs[key]
oof["query_set"] = "tuning"
pd.concat([oof, out]).to_parquet(PROJECT_ROOT / "cache" / "phase9_predictions.parquet", index=False)
comparison.round(4).to_csv(PROJECT_ROOT / "experiments" / "phase9_model_comparison.tsv", sep="\t", lineterminator="\n")
print(f"saved predictions of '{best}'  (total notebook time {time.time() - start:.0f}s)")

saved predictions of 'gradient boosting'  (total notebook time 489s)


---
# Summary — Phase 9

| model | tuning F0.5 (out-of-fold) | validation F0.5 | pair precision | pair recall | time |
|---|---|---|---|---|---|
| Jaccard rule (Phase 7) | — | 0.706 | 0.79 | 0.67 | — |
| logistic regression | 0.887 | 0.889 | 0.954 | 0.811 | 53 s |
| random forest | 0.914 | 0.917 | 0.969 | 0.848 | 116 s |
| **gradient boosting** | **0.932** | **0.934** | **0.979** | **0.871** | 204 s |

- Same candidates, same features for all models — the **model** alone moves the score from 0.706 to **0.934**.
- Tuning (out-of-fold) and validation scores agree within 0.002 → the evaluation is honest and the model does not overfit.
- Of 61,436 pairs the model predicts as matches, only **1,322 are false merges**.
- The ceiling is candidate recall (0.94): pair recall 0.871 means the model finds ~93% of the matches that blocking delivered.

**Next (Phase 10):** a closer look at the decision rule — per-entity decisions, the "each S2/S3 record belongs to at most one S1" rule,
and whether a smarter rule than one global threshold helps.